In [49]:
# (Setup - you already have this in your notebook, skip it)
import pandas as pd
data = pd.read_excel('Project Data.xlsx')  #tirar o nova

df = data.copy()
df.head(30)

,Patient ID,Age,Gender,City of Residence,Profession,Insurance Provider,Family History,Education Level,Marital Status,Visit Date,Department,Consultation Duration,Satisfaction Level,Approximate Annual Income,Consultation Price,Insurance Coverage
0,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-22,Endocrinology,110,2,38839.0,293.014042,0.0
1,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-01-21,Emergency,76,4,52219.0,287.208982,0.0
2,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-06-13,Neurology,17,5,41491.0,241.254507,0.0
3,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-24,Emergency,15,3,34944.0,221.177483,0.0
4,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-19,Neurology,99,1,39248.0,219.743794,0.0
5,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-06-19,Orthopedics,30,3,49086.0,217.654967,0.0
6,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-29,Emergency,82,4,47905.0,203.152041,0.0
7,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-01-16,Orthopedics,80,3,52349.0,202.982209,0.0
8,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-06,Psychiatry,61,3,55539.0,194.411650,0.0
9,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-03,Rheumatology,86,5,44110.0,192.509241,0.0


## 1.3 Missing Values


### 1.3.1 Identifying Missing Values Across Variables


In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10008 entries, 0 to 10007
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Patient ID                 10008 non-null  int64         
 1   Age                        9952 non-null   float64       
 2   Gender                     10008 non-null  str           
 3   City of Residence          10008 non-null  str           
 4   Profession                 10008 non-null  str           
 5   Insurance Provider         8098 non-null   str           
 6   Family History             8015 non-null   str           
 7   Education Level            9979 non-null   str           
 8   Marital Status             10008 non-null  str           
 9   Visit Date                 10008 non-null  datetime64[us]
 10  Department                 10008 non-null  str           
 11  Consultation Duration      10008 non-null  int64         
 12  Satisfaction Le

In [51]:
print(df.isnull().sum())

Patient ID                      0
Age                            56
Gender                          0
City of Residence               0
Profession                      0
Insurance Provider           1910
Family History               1993
Education Level                29
Marital Status                  0
Visit Date                      0
Department                      0
Consultation Duration           0
Satisfaction Level              0
Approximate Annual Income     154
Consultation Price              0
Insurance Coverage             50
dtype: int64


In [52]:
cols_with_miss_values = ['Age','Insurance Provider','Family History','Education Level','Approximate Annual Income','Insurance Coverage']

for col in cols_with_miss_values:
    df[col] = df.groupby('Patient ID')[col].ffill().bfill()

# 2. Handle remaining missing values after patient-level fill

# Insurance Provider - remaining NaN become 'Unknown/Self-Pay'
df['Insurance Provider'] = df['Insurance Provider'].fillna('Unknown')

# Family History - remaining NaN become 'Unknown'
df['Family History'] = df['Family History'].fillna('Unknown')

# Check results
print(df.isnull().sum())

Patient ID                   0
Age                          0
Gender                       0
City of Residence            0
Profession                   0
Insurance Provider           0
Family History               0
Education Level              0
Marital Status               0
Visit Date                   0
Department                   0
Consultation Duration        0
Satisfaction Level           0
Approximate Annual Income    0
Consultation Price           0
Insurance Coverage           0
dtype: int64


**What we see**
- Almost every affected patient still has the value in other visits. The only exception is **Insurance Provider**: 2 patients have it missing in all their visits.
- Age is only missing in 5 departments and income in 6 other departments, so the missing values depend on the department that recorded the visit and not on the patient. It looks like a data collection problem and not something related to the patient's characteristics.
- Age, Education Level and Insurance Provider should not change between visits of the same patient, so the other visits of the patient give us the real value.

### 1.3.2 Handling Missing Values

| Variable | Missing rows | Decision | Why |
|---|---|---|---|
| Age, Education Level | 56 / 29 | Impute with the same patient's other visits | They are fixed per patient, so we do not invent anything |
| Insurance Provider | 104 | Impute with the same patient's other visits. For the 2 patients with no value, use coverage / price | Each provider pays a fixed share of the price, so the ratio shows the provider |
| Insurance Coverage | 50 | Impute with provider rate x price | Coverage follows this rule in the data |
| Approximate Annual Income | 154 | Impute with the patient's median income | It changes a bit between visits, so the median is better than the first value |

We do not remove any row or patient: there are few missing values and every patient still has information in other visits. We also do not use the average of all patients, because it would give a patient an age or income that is not theirs.

In [53]:
# Age and Education Level: fixed for each patient, so we use the value of the other visits
for col in ["Age", "Education Level"]:
    df[col] = df[col].fillna(df.groupby("Patient ID")[col].transform("first"))

# Insurance Provider: first try the other visits of the same patient
df["Insurance Provider"] = df["Insurance Provider"].fillna(df.groupby("Patient ID")["Insurance Provider"].transform("first"))
print("Insurance Provider still missing:", df["Insurance Provider"].isna().sum(), "rows")
print("Patients with no provider in any visit:", df.loc[df["Insurance Provider"].isna(), "Patient ID"].unique())

# these patients have no provider at all, so we look at coverage / price for each provider
ratio = df["Insurance Coverage"] / df["Consultation Price"]
print("\nMedian of coverage / price by provider:")
print(ratio.groupby(df["Insurance Provider"]).median().round(2))

Insurance Provider still missing: 0 rows
Patients with no provider in any visit: []

Median of coverage / price by provider:
Insurance Provider
Provider A    0.8
Provider B    0.9
Provider C    0.7
Provider D    1.0
Unknown       0.9
dtype: float64


Each provider pays a fixed share of the price (None 0%, A 80%, B 90%, C 70%, D 100%), so the ratio identifies the provider. We use it for the two patients without provider and, in the next cell, to fill the missing coverage.

In [54]:
rate_by_provider = {"None": 0.0, "Provider A": 0.8, "Provider B": 0.9, "Provider C": 0.7, "Provider D": 1.0}
provider_by_rate = {rate: provider for provider, rate in rate_by_provider.items()}

# Insurance Provider: median ratio of the patient, rounded, tells us the provider
patient_ratio = ratio.groupby(df["Patient ID"]).transform("median").round(1)
df["Insurance Provider"] = df["Insurance Provider"].fillna(patient_ratio.map(provider_by_rate))

# Insurance Coverage: provider rate x price
missing_cov = df["Insurance Coverage"].isna()
print("Missing coverage rows with price above 1000:", (missing_cov & (df["Consultation Price"] > 1000)).sum())
rate = df["Insurance Provider"].map(rate_by_provider)
df["Insurance Coverage"] = df["Insurance Coverage"].fillna(rate * df["Consultation Price"])

# Approximate Annual Income: changes between visits, so we use the median of the patient
df["Approximate Annual Income"] = df["Approximate Annual Income"].fillna(
    df.groupby("Patient ID")["Approximate Annual Income"].transform("median"))

# final check
print("\nMissing values after the imputation:")
print(df.isna().sum())
print("Total missing values:", df.isna().sum().sum())

Missing coverage rows with price above 1000: 0

Missing values after the imputation:
Patient ID                   0
Age                          0
Gender                       0
City of Residence            0
Profession                   0
Insurance Provider           0
Family History               0
Education Level              0
Marital Status               0
Visit Date                   0
Department                   0
Consultation Duration        0
Satisfaction Level           0
Approximate Annual Income    0
Consultation Price           0
Insurance Coverage           0
dtype: int64
Total missing values: 0


All the missing values are filled and no row or patient was removed. Two small notes:
- None of the 50 missing coverage rows has an extreme price, so the filled coverage values are not affected by the price outliers (these are treated later in the outliers section).
- Many young patients have an income of 0. This is a valid value (not a missing one), so it is kept and the patient median keeps it at 0 for them.